# 00 - Generate and inspect the documents

## Goal

Most real RAG projects don't get clean text. This corpus mixes formats on purpose:

- Markdown policies and an incident postmortem
- CSV financial rows (some questions need math)
- HTML prose with spec tables
- nested JSON support tickets
- plain-text meeting minutes with negations and decisions that change
- a PDF safety bulletin generated in the next cell

Everything is made up, so it's safe to show and there's an answer key for evaluation.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
core_notebook_path = ROOT / "notebooks/_rag_demo_core.ipynb"
get_ipython().run_line_magic("run", f'"{core_notebook_path}"')
DATA = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"
print(f"Project root: {ROOT}")
openai_settings = get_openai_settings(require_api_key=False)
print(f"OpenAI configured: {openai_settings.configured}; "
      f"embedding={openai_settings.embedding_model}; chat={openai_settings.chat_model}")


Project root: /Users/snomula/Downloads/sripal/Context Engineern-RAG
OpenAI configured: False; embedding=text-embedding-3-small; chat=gpt-5-mini


In [2]:
pdf_path = generate_pdf(DATA / "raw/northstar_safety_bulletin.pdf")
print(f"Generated {pdf_path.name}: {pdf_path.stat().st_size:,} bytes")


Generated northstar_safety_bulletin.pdf: 1,388 bytes


In [3]:
documents = load_corpus(DATA / "raw")
for doc in documents:
    print(f"{doc.doc_id:30} {doc.metadata['doc_type']:10} "
          f"{len(doc.text.split()):4} words  {len(doc.elements):2} structural elements")


incident-search-2026-01-22     markdown    221 words   6 structural elements
handbook-2026                  markdown    281 words   6 structural elements
safety-bulletin-2026-04        pdf         117 words   1 structural elements
ops-minutes-2026-02-03         text        198 words   7 structural elements
manual-orion-x2                html        234 words  10 structural elements
financials-2025                csv         194 words   9 structural elements
support-tickets-2026-02        json        147 words   3 structural elements


### Look at the structure, not just the text

The loaders keep rows, records, headings and pages as separate elements. Chunking in the next notebook depends on that.

In [4]:
for doc in documents:
    print("\n", "=" * 80, "\n", doc.doc_id)
    for element in doc.elements[:2]:
        print({k: v for k, v in element.items() if k not in {"data", "text"}})
        print(element["text"][:260].replace("\n", " "))



 incident-search-2026-01-22
{'type': 'section', 'heading': 'Postmortem: Search API latency on 2026-01-22', 'level': 1}
Document ID: `incident-search-2026-01-22`   Classification: Internal   Severity: SEV-2
{'type': 'section', 'heading': 'Summary', 'level': 2}
From 14:07 to 14:49 UTC, the customer support search API had a p95 latency of 8.2 seconds. Approximately 31% of search requests exceeded the two-second service objective. No customer data was lost.

 handbook-2026
{'type': 'section', 'heading': 'Northstar Mobility Employee Handbook', 'level': 1}
Document ID: `handbook-2026`   Version: 3.2   Effective date: 2026-01-15   Owner: People Operations
{'type': 'section', 'heading': 'Hybrid work', 'level': 2}
Employees assigned to the Seattle office may work remotely up to **three days per week**. Tuesday is the common collaboration day and requires in-office attendance unless a director approves an exception in writing. Employees who live more than 80 miles from a

 safety-bulletin-2026-

In [5]:
import json
cases = [json.loads(line) for line in (DATA / "eval/questions.jsonl").read_text().splitlines()]
print(f"Evaluation cases: {len(cases)}")
for case in cases:
    print(f"{case['id']} [{case['category']}] {case['question']}")


Evaluation cases: 10
q01 [policy] How many days per week may a Seattle employee work remotely?
q02 [policy] What must an employee do after suspected credential compromise, and how quickly?
q03 [table_lookup] What torque is required for the Orion X2 rear axle nut?
q04 [table_lookup] For an E17 error, what should a technician inspect first?
q05 [table_calculation] What was Q4 North America Orion Scooter gross margin in dollars?
q06 [json_record] Which ticket links heavy rain with E17 and what fixed it?
q07 [negation] Did Northstar approve extending the Lyra Battery warranty to 30 months?
q08 [multi_fact] What caused the January 22 search latency incident?
q09 [pdf] What is the battery isolation distance in the safety bulletin?
q10 [multi_document] Can I use the Q4 figures in public material without qualification?


## Takeaway

Parsing affects retrieval quality. If a loader flattens a table or splits a value away from its row header, no embedding model can put that relationship back.